# Automated planning (STRIPS, PDDL) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Search over symbolic fact sets
STRIPS actions are preconditions plus add and delete effects.

In [ ]:
state=frozenset(['AtA','RoadAB']); pre={'AtA','RoadAB'}; add={'AtB'}; dele={'AtA'}; new=(set(state)-dele)|add; plt.figure(figsize=(5,3)); plt.bar(['before','after'],[len(state),len(new)]); plt.title('apply action')
assert new=={'RoadAB','AtB'}

In [ ]:
state=frozenset(['AtA']); acts={'MoveAB':{'AtA','RoadAB'},'Wait':{'AtA'}}; app=[n for n,p in acts.items() if p<=state]; plt.figure(figsize=(5,3)); plt.bar(list(acts),[n in app for n in acts]); plt.title('preconditions')
assert app==['Wait']

In [ ]:
from collections import deque
acts=[('MoveAB',{'AtA','RoadAB'},{'AtB'},{'AtA'}),('MoveBC',{'AtB','RoadBC'},{'AtC'},{'AtB'})]; start=frozenset(['AtA','RoadAB','RoadBC']); goal={'AtC'}; q=deque([start]); par={start:(None,None)}
while q:
 s=q.popleft()
 if goal<=s: break
 for n,pre,add,dele in acts:
  if pre<=s:
   ns=frozenset((set(s)-dele)|add)
   if ns not in par: par[ns]=(s,n); q.append(ns)
plan=[]; x=s
while par[x][0] is not None: old,a=par[x]; plan.append(a); x=old
plt.figure(figsize=(5,3)); plt.bar(['plan length'],[len(plan)]); plt.title('forward planning')
assert plan[::-1]==['MoveAB','MoveBC']

In [ ]:
states=[frozenset(['AtA']),frozenset(['AtB']),frozenset(['AtC']),frozenset(['AtC','HasBox'])]; goal={'AtC','HasBox'}; h=[len(goal-set(s)) for s in states]; plt.figure(figsize=(5,3)); plt.bar(range(4),h); plt.title('unsatisfied goals')
assert h==[2,2,1,0]

In [ ]:
acts=[('MoveAB',{'AtA','RoadAB'},{'AtB'},{'AtA'}),('MoveBC',{'AtB','RoadBC'},{'AtC'},{'AtB'})]; subgoal={'AtC'}; n,pre,add,dele=acts[1]; reg=(subgoal-add)|pre; plt.figure(figsize=(5,3)); plt.bar(['goal','regressed'],[len(subgoal),len(reg)]); plt.title('regression')
assert reg=={'AtB','RoadBC'}